In [ ]:
#here in this we will finetune the llama-3.2-1b-instruct for summarization purposes

In [1]:
%pip install -U transformers accelerate peft trl datasets sentencepiece bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [22]:
#importing all required libraries and modules



from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

import torch
from peft import PeftModel

In [3]:
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [5]:
dataset['train'][0]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [6]:
dataset['train']

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 287113
})

In [7]:

#now lets make the function for the formatting the dataset

def format_dataset(example):

  prompt=f"""###Instruction
  {example['article']}

  ### Response
  {example['highlights']}"""
  return {'text':prompt}

In [8]:
dataset=dataset['train'].map(format_dataset)

Map:   0%|          | 0/287113 [00:00<?, ? examples/s]

In [9]:
dataset=dataset.select(range(1000))

In [10]:
dataset

Dataset({
    features: ['article', 'highlights', 'id', 'text'],
    num_rows: 1000
})

In [ ]:
#now lets load the tokenizer

model_name='meta-llama/Llama-3.2-1B-Instruct'

tokenizer=AutoTokenizer.from_pretrained(model_name)


In [ ]:
#now lets create the config for the quantization of 4bit of our model
quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.bfloat16,
                                       bnb_4bit_use_double_quant=True)


In [ ]:
#now lets load the quantized LLm

quantized_llm=AutoModelForCausalLM.from_pretrained(model_name,quantization_config=quantization_config,device_map="auto",
    torch_dtype=torch.bfloat16 #device_map='auto' Automatically place the model on the available GPU
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
#now we have loaded the quantized LLM
print(quantized_llm)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

In [ ]:
#lets count the trainable parameters from this LLM
total_params = sum(p.numel() for p in quantized_llm.parameters())
trainable_params = sum(p.numel() for p in quantized_llm.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 749,275,136
Trainable Parameters: 262,735,872


In [ ]:
#here we can see trainable parameters lets freeze it

quantized_llm = prepare_model_for_kbit_training(quantized_llm)

In [ ]:
#lets count the trainable parameters from this LLM
total_params = sum(p.numel() for p in quantized_llm.parameters())
trainable_params = sum(p.numel() for p in quantized_llm.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 749,275,136
Trainable Parameters: 0


In [16]:
#now we have add the adapters in this freeze quantized LLM
#here we are finetuning using LoRA technique
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [ ]:
#now lets attach these adapters in the quantized LLM

model = get_peft_model(quantized_llm, lora_config)

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.bfloat16)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
tokenizer.pad_token = tokenizer.eos_token #here we have added <eos> as a padding tokens for handling the differnt length of inputs in batch

In [ ]:
#now lets configure the arguments for retraining
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/llama_using_LoRA_summarization",

    num_train_epochs=3,

    per_device_train_batch_size=4,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    logging_steps=10,
    save_strategy="steps",
save_steps=25,          # save ~5 times per epoch given your data size
save_total_limit=2,     # keep only last 2 checkpoints so Drive doesn't fill up



    fp16=False,
    bf16=True,  #before fp16=True ad bf16=False

    optim="paged_adamw_8bit",

    lr_scheduler_type="cosine",

    warmup_ratio=0.03,

    report_to="none",
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
#lets create the trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset, #dataset for finetuning #it directly contains the data no need to do dataset['train']
    processing_class=tokenizer,
    args=training_args,
    formatting_func=lambda example: example["text"], #input for the LLM
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Applying formatting function to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:

#lets do fine-tuning
#but initially we have to do it
# trainer.train()
trainer.train(resume_from_checkpoint=True) #if our session got disconnected then it will again retrain from where it stopped


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
180,2.271821
190,2.268240
200,2.241587
210,2.238021
220,2.269145
230,2.188840
240,2.242808
250,2.237458
260,2.133765
270,2.099898


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=375, training_loss=1.1535229848225912, metrics={'train_runtime': 6622.7044, 'train_samples_per_second': 0.453, 'train_steps_per_second': 0.057, 'total_flos': 1.7222272222937088e+16, 'train_loss': 1.1535229848225912, 'entropy': 2.182366704940796, 'num_tokens': 1182695.0, 'mean_token_accuracy': 0.5208694159984588, 'epoch': 3.0})

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
model_name='meta-llama/Llama-3.2-1B-Instruct'

In [23]:
#lets load the quantized llm


model_name = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model_for_lora=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [24]:
# #now lets load this finetuned LLM
adapter_path='/content/drive/MyDrive/llama_using_LoRA_summarization/checkpoint-375'
fine_tuned_model = PeftModel.from_pretrained(model_for_lora, adapter_path)

print('Fine_tuned_llm_loaded')

Fine_tuned_llm_loaded


In [36]:
tokenizer=AutoTokenizer.from_pretrained(adapter_path) #along with adapter it also stores the tokenizer

In [27]:
#lets load the dataset
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")


In [28]:
dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [29]:
#we will use the first 10 rows for the testing purpose only from the testing set
testing_dataset=dataset['test']
print(testing_dataset)

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 11490
})


In [30]:
testing_dataset=testing_dataset.select(range(10))

In [26]:
def format_dataset(example):

  prompt=f"""###Instruction
  {example['article']}

  ### Response
  """
  return {'testing_text':prompt}

In [31]:
testing_dataset=testing_dataset.map(format_dataset)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [32]:
testing_dataset['testing_text']

Column(['###Instruction\n  (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC\'s founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians\' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speakin

In [40]:
#lets test it and compare the reference with the generated summary

def summarizer(data):
      prompt=data['testing_text']
      inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned_model.device)

      with torch.no_grad():
          outputs = fine_tuned_model.generate(
              **inputs,
              max_new_tokens=128,
              do_sample=False,
              pad_token_id=tokenizer.eos_token_id,
              repetition_penalty=1.1,
          )

      generated = outputs[0][inputs["input_ids"].shape[1]:]
      return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Test it!
count=1
for row in testing_dataset:
    print(f"Reference Summary {count}: {row['highlights']} \n")
    print(f"Generated Summary {count}: {summarizer(row)}\n")
    count+=1
    print('\n')

Reference Summary 1: Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis . 

Generated Summary 1: NEW: Palestinians sign Rome Statute, become official ICC member .
NEW: ICC opens preliminary examination into alleged war crimes in Gaza .
Israel and U.S. opposed Palestinians' bid to join the court .



Reference Summary 2: Theia, a bully breed mix, was apparently hit by a car, whacked with a hammer and buried in a field .
"She's a true miracle dog and she deserves a good life," says Sara Mellado, who is looking for a home for Theia . 

Generated Summary 2: Dog appears to have died after being struck by car, then buried .
Dog had leg injuries, dislocated jaw, caved-in sinus cavity .
Veterinary hospital says it has received donations to help with her care .



Reference Summary 3: Mohammad Javad Zarif has spent m

In [41]:
#now lets add this generated summary in the testing dataset and then we will use it for evaluation

In [42]:
testing_dataset

Dataset({
    features: ['article', 'highlights', 'id', 'testing_text'],
    num_rows: 10
})

In [43]:
import torch

def generate_summary(dataset, model, tokenizer):

    count=1
    generated_responses = []

    for sample in dataset:

        prompt = sample["testing_text"]   # Uses the same template used during training

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated = outputs[0][inputs["input_ids"].shape[1]:]
        print("Response: ",count,'generated')
        response = tokenizer.decode(
            generated,
            skip_special_tokens=True
        ).strip()

        generated_responses.append(response)
        count+=1

    dataset = dataset.add_column(
        "generated_response",
        generated_responses
    )

    return dataset

In [44]:
testing_dataset=generate_summary(testing_dataset,fine_tuned_model,tokenizer)

Response:  1 generated
Response:  2 generated
Response:  3 generated
Response:  4 generated
Response:  5 generated
Response:  6 generated
Response:  7 generated
Response:  8 generated
Response:  9 generated
Response:  10 generated


In [45]:
testing_dataset

Dataset({
    features: ['article', 'highlights', 'id', 'testing_text', 'generated_response'],
    num_rows: 10
})

In [46]:
!pip install evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=4c69e158ab5d3f3a436e958a3fb3a4a26a63b4b1bf56d9c161d07b03af11160a
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [47]:
import evaluate

rouge = evaluate.load("rouge")

In [48]:
#now lets do evaluation using the ROUGE metrics
def calculate_rouge(dataset):
    """
    Calculate ROUGE-1, ROUGE-2 and ROUGE-L scores.

    Args:
        dataset: HuggingFace Dataset containing:
                 - highlights (reference summaries)
                 - generated_response (predicted summaries)

    Returns:
        Dictionary containing ROUGE scores.
    """

    predictions = dataset["generated_response"]
    references = dataset["highlights"]

    scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    print("=" * 50)
    print("ROUGE Evaluation")
    print("=" * 50)
    print(f"ROUGE-1 : {scores['rouge1']:.4f}")
    print(f"ROUGE-2 : {scores['rouge2']:.4f}")
    print(f"ROUGE-L : {scores['rougeL']:.4f}")

    return scores

In [50]:
rouge_scores = calculate_rouge(testing_dataset)

ROUGE Evaluation
ROUGE-1 : 0.3858
ROUGE-2 : 0.1637
ROUGE-L : 0.2604


In [51]:
rouge_scores

{'rouge1': np.float64(0.38578985877757144),
 'rouge2': np.float64(0.1637235011573643),
 'rougeL': np.float64(0.2604202402819914),
 'rougeLsum': np.float64(0.35670432436389465)}